In [1]:
!pip install -q langchain-community openai faiss-cpu chromadb tiktoken
!pip install -q google-generativeai pymupdf
!pip install -q langchain-chroma langchain-openai langgraph langchain-core

In [2]:
# ============================================================================
# PROCESADOR MÚLTIPLE DE PAPERS - VECTORES DUALES
# ============================================================================

import os
import re
import json
import pickle
import warnings
from typing import TypedDict, List, Dict, Any
from pathlib import Path

# PDF Processing
import fitz  # pymupdf

# LangChain Core
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain.schema import Document as LCDocument

# APIs externas
import google.generativeai as genai
from google.colab import userdata, drive

warnings.filterwarnings('ignore')

In [3]:
# Montar Drive
print("📁 Montando Google Drive...")
drive.mount('/content/drive')

# Configurar APIs
print("🔑 Configurando APIs...")
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
genai.configure(api_key=userdata.get('GEMINI_API_KEY_3'))

# Inicializar modelos
print("🤖 Inicializando modelos...")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()
gemini_model = genai.GenerativeModel('gemini-2.0-flash')

print("✅ APIs configuradas correctamente!")
print("✅ Modelos listos para usar:")
print(f"  - LLM: {llm.model_name}")
print(f"  - Embeddings: OpenAI")
print(f"  - Gemini: gemini-2.0-flash")

# Test rápido de conexiones
print("\n🧪 Probando conexiones...")
try:
    # Test OpenAI
    test_embedding = embeddings.embed_query("test")
    print("✅ OpenAI: Conectado")
except Exception as e:
    print(f"❌ OpenAI: Error - {e}")

try:
    # Test Gemini
    test_response = gemini_model.generate_content("Di solo 'OK' si me recibes")
    print(f"✅ Gemini: Conectado - Respuesta: {test_response.text.strip()}")
except Exception as e:
    print(f"❌ Gemini: Error - {e}")

print("\n🎯 Variables globales disponibles:")
print("  - llm")
print("  - embeddings")
print("  - gemini_model")
print("\n✅ Listo para ejecutar el procesamiento de papers!")

📁 Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔑 Configurando APIs...
🤖 Inicializando modelos...
✅ APIs configuradas correctamente!
✅ Modelos listos para usar:
  - LLM: gpt-4o-mini
  - Embeddings: OpenAI
  - Gemini: gemini-2.0-flash

🧪 Probando conexiones...
✅ OpenAI: Conectado
✅ Gemini: Conectado - Respuesta: OK

🎯 Variables globales disponibles:
  - llm
  - embeddings
  - gemini_model

✅ Listo para ejecutar el procesamiento de papers!


In [4]:
# ============================================================================
# FUNCIONES DE PROCESAMIENTO
# ============================================================================

def sanitize_collection_name(name: str) -> str:
    """Sanitiza nombres para compatibilidad con Chroma"""
    # Remover caracteres especiales y reemplazar con guiones bajos
    sanitized = re.sub(r'[^a-zA-Z0-9._-]', '_', name)

    # Remover múltiples guiones bajos consecutivos
    sanitized = re.sub(r'_+', '_', sanitized)

    # Asegurar que comience y termine con alfanumérico
    sanitized = sanitized.strip('_.-')

    # Si está vacío o muy corto, usar nombre genérico
    if len(sanitized) < 3:
        sanitized = f"paper_{hash(name) % 10000}"

    # Limitar longitud a 50 caracteres para seguridad
    if len(sanitized) > 50:
        sanitized = sanitized[:50].rstrip('_.-')

    return sanitized

def extract_pdf_text(pdf_path: str) -> tuple:
    """Extrae texto del PDF con metadata básica"""
    try:
        doc = fitz.open(pdf_path)
        full_text = "\n".join([page.get_text() for page in doc])

        metadata = {
            'total_pages': len(doc),
            'total_chars': len(full_text),
            'filename': pdf_path,
            'paper_name': Path(pdf_path).stem
        }

        doc.close()
        return full_text, metadata
    except Exception as e:
        print(f"❌ Error procesando {pdf_path}: {e}")
        return None,

def extract_title_from_text(text: str, gemini_model) -> str:
    """Extrae el título del paper usando Gemini"""
    try:
        # Tomar las primeras líneas del documento para encontrar el título
        first_part = text[:2000]  # Primeros 2000 caracteres

        prompt = f"""Analiza el siguiente texto del inicio de un artículo científico y extrae únicamente el TÍTULO principal del paper.

Texto:
{first_part}

Instrucciones:
- Identifica y extrae solo el título principal del artículo
- NO incluyas nombres de autores, afiliaciones, resumen, o cualquier otro contenido
- Devuelve únicamente el título, sin comillas ni prefijos
- Si hay múltiples líneas que parecen título, combínalas en una sola línea
- Si no puedes identificar un título claro, devuelve "Título no identificado"

TÍTULO:"""

        response = gemini_model.generate_content(prompt)
        title = response.text.strip()

        # Limpiar el título de posibles prefijos
        title = re.sub(r'^(TÍTULO|Title|TITLE):\s*', '', title, flags=re.IGNORECASE)
        title = title.strip('"').strip("'").strip()

        if not title or len(title) < 5:
            return "Título no identificado"

        return title

    except Exception as e:
        print(f"⚠️ Error extrayendo título: {e}")
        return "Título no identificado"


def create_analysis_prompts(text: str) -> Dict[str, str]:
    """Crea prompts para análisis de secciones y referencias"""
    base_text = f"Analiza el siguiente texto de un artículo científico:\n\n{text}"

    return {
        'sections': f"""{base_text}

Identifica y lista todos los encabezados de sección principales (como Introduction, Methods, Results, Discussion, Conclusion, etc.) en el orden que aparecen.

Formato de respuesta:
## SECCIONES IDENTIFICADAS:
- [Lista de secciones en orden]

Solo los nombres de las secciones, sin explicaciones adicionales.""",

        'references': f"""{base_text}

Identifica y extrae todas las referencias bibliográficas del artículo.

Formato de respuesta:
## REFERENCIAS BIBLIOGRÁFICAS:
- [Número] Referencia completa"""
    }

def parse_gemini_sections(response_text: str) -> List[str]:
    """Extrae secciones de respuesta de Gemini"""
    sections = []
    if "## SECCIONES IDENTIFICADAS:" in response_text:
        sections_text = response_text.split("## SECCIONES IDENTIFICADAS:")[1]
        if "## REFERENCIAS BIBLIOGRÁFICAS:" in sections_text:
            sections_text = sections_text.split("## REFERENCIAS BIBLIOGRÁFICAS:")[0]

        for line in sections_text.split('\n'):
            line = line.strip()
            if line.startswith('- '):
                section = line[2:].strip()
                if section:
                    sections.append(section)
    return sections

def parse_references_to_dict(text: str) -> dict:
    """Convierte referencias en diccionario {numero: texto}"""
    matches = list(re.finditer(r"- \[(\d+)\] (.+?)(?=(?:- \[\d+\])|\Z)", text, re.DOTALL))
    return {match.group(1).strip(): match.group(2).strip().replace("\n", " ").replace("  ", " ")
            for match in matches}

def extract_references(text: str) -> List[str]:
    """Extrae números de referencias citadas del texto"""
    raw_refs = re.findall(r"\[([^\[\]]+?)\]", text)
    final_refs = set()

    for group in raw_refs:
        parts = [p.strip() for p in group.split(',')]
        for part in parts:
            if '–' in part or '-' in part:
                sep = '–' if '–' in part else '-'
                try:
                    start, end = map(int, part.split(sep))
                    final_refs.update(str(i) for i in range(start, end + 1))
                except ValueError:
                    continue
            else:
                if part.isdigit():
                    final_refs.add(part)

    return sorted(final_refs, key=int)

def split_text_by_sections(full_text: str, section_titles: List[str], resolved_references: dict) -> List[Dict]:
    """Divide el texto por secciones y asocia referencias"""
    sections_data = []
    text_with_markers = full_text + "\n## END_OF_DOCUMENT##"

    # Crear patrón regex para títulos de sección
    escaped_titles = [re.escape(title) for title in section_titles]
    section_pattern = '|'.join([f'^{title}' for title in escaped_titles])
    end_pattern = r'^## END_OF_DOCUMENT##'
    combined_pattern = f'(?m){section_pattern}|{end_pattern}'

    all_matches = list(re.finditer(combined_pattern, text_with_markers))

    if not all_matches or all_matches[0].group(0).strip() == "## END_OF_DOCUMENT##":
        print("⚠️ No se encontraron secciones")
        return []

    for i in range(len(all_matches) - 1):
        start_match = all_matches[i]
        end_match = all_matches[i + 1]

        title = start_match.group(0).strip()
        original_title = next((t for t in section_titles if t.strip().lower() == title.lower()), title)

        start_pos = start_match.end()
        end_pos = end_match.start()
        section_text = text_with_markers[start_pos:end_pos].strip()

        # Extraer referencias citadas
        cited_refs = extract_references(section_text)
        section_resolved_refs = {
            ref_id: resolved_references.get(ref_id, f"Reference [{ref_id}] not found.")
            for ref_id in cited_refs
        }

        sections_data.append({
            "title": original_title,
            "text": section_text,
            "refs": cited_refs,
            "resolved_refs": section_resolved_refs
        })

    return sections_data

def create_chunks_for_chroma(sections_data: List[Dict], paper_name: str, paper_title: str) -> List[Dict]:
    """Crea chunks finos por párrafo para Chroma"""
    chunks = []
    for section in sections_data:
        paragraphs = section['text'].split('\n\n')
        for i, paragraph in enumerate(paragraphs):
            if len(paragraph.strip()) > 50:
                chunk_refs = extract_references(paragraph)
                chunk_resolved_refs = {
                    ref_id: section['resolved_refs'].get(ref_id, f"Ref [{ref_id}] not found")
                    for ref_id in chunk_refs
                }

                chunks.append({
                    "text": paragraph.strip(),
                    "metadata": {
                        "paper_name": paper_name,
                        "paper_title": paper_title,  # ✅ LÍNEA AÑADIDA
                        "section_title": section['title'],
                        "chunk_id": f"{paper_name}_{section['title']}_{i}",
                        "references_mentioned": chunk_refs,
                        "resolved_references": chunk_resolved_refs,
                        "chunk_type": "paragraph"
                    }
                })
    return chunks

def create_chunks_for_faiss(sections_data: List[Dict], paper_name: str, paper_title: str) -> List[Dict]:
    """Crea chunks por sección completa para FAISS"""
    return [{
        "text": section['text'],
        "metadata": {
            "paper_name": paper_name,
            "paper_title": paper_title,  # ✅ LÍNEA AÑADIDA
            "section_title": section['title'],
            "chunk_id": f"{paper_name}_{section['title']}",
            "references_mentioned": section['refs'],
            "resolved_references": section['resolved_refs'],
            "chunk_type": "full_section"
        }
    } for section in sections_data]

def clean_metadata_for_chroma(metadata: Dict) -> Dict:
    """Limpia metadatos para compatibilidad con Chroma"""
    cleaned = {}
    for key, value in metadata.items():
        if isinstance(value, list):
            cleaned[key] = ", ".join(map(str, value)) if value else ""
        elif isinstance(value, dict):
            cleaned[key] = json.dumps(value) if value else "{}"
        else:
            cleaned[key] = str(value)
    return cleaned

# ============================================================================
# PROCESAMIENTO INDIVIDUAL DE PAPERS
# ============================================================================

def process_single_paper(pdf_path: str, output_dir: str) -> Dict[str, Any]:
    """Procesa un paper individual y genera sus vectores duales"""

    paper_name = Path(pdf_path).stem
    # Crear nombre sanitizado para colecciones
    sanitized_name = sanitize_collection_name(paper_name)

    print(f"\n🔄 Procesando: {paper_name}")
    print(f"📝 Nombre sanitizado: {sanitized_name}")

    # 1. Extraer texto
    full_text, metadata = extract_pdf_text(pdf_path)
    if full_text is None:
        return {"status": "error", "paper_name": paper_name, "error": "No se pudo extraer texto"}

    print(f"📊 Páginas: {metadata['total_pages']}, Caracteres: {metadata['total_chars']}")

    # 2. Extraer título usando Gemini
    try:
        print("📋 Extrayendo título del paper...")
        paper_title = extract_title_from_text(full_text, gemini_model)
        print(f"📋 Título extraído: {paper_title}")
    except Exception as e:
        print(f"⚠️ Error extrayendo título: {e}")
        paper_title = "Título no identificado"

    # 3. Análisis con Gemini
    try:
        print("🧠 Iniciando análisis con Gemini...")
        prompts = create_analysis_prompts(full_text)
        print("📝 Prompts creados, enviando a Gemini...")

        # Obtener secciones
        print("  📋 Solicitando secciones...")
        sections_response = gemini_model.generate_content(prompts['sections'])
        print("  📋 Respuesta de secciones recibida, procesando...")
        sections = parse_gemini_sections(sections_response.text)
        print(f"  📋 Secciones procesadas: {len(sections)}")

        # Obtener referencias
        print("  📚 Solicitando referencias...")
        references_response = gemini_model.generate_content(prompts['references'])
        print("  📚 Respuesta de referencias recibida, procesando...")
        resolved_references = parse_references_to_dict(references_response.text)
        print(f"  📚 Referencias procesadas: {len(resolved_references)}")

        print(f"✅ Análisis completado - Secciones: {len(sections)}, Referencias: {len(resolved_references)}")

    except Exception as e:
        print(f"❌ Error en análisis Gemini: {e}")
        print(f"🔍 Tipo de error: {type(e).__name__}")
        import traceback
        print(f"🔍 Traceback: {traceback.format_exc()}")
        return {"status": "error", "paper_name": paper_name, "error": f"Error Gemini: {e}"}

    # 4. Procesar secciones
    print("📄 Procesando secciones...")
    sections_data = split_text_by_sections(full_text, sections, resolved_references)

    # 5. Crear chunks
    print("✂️ Creando chunks...")
    chroma_chunks = create_chunks_for_chroma(sections_data, paper_name, paper_title)
    faiss_chunks = create_chunks_for_faiss(sections_data, paper_name, paper_title)

    print(f"📦 Chunks Chroma: {len(chroma_chunks)}, FAISS: {len(faiss_chunks)}")

    # 6. Crear vectorstores
    try:
        print("🔄 Creando vectorstores...")
        # Chroma - usar nombre sanitizado
        chroma_texts = [chunk["text"] for chunk in chroma_chunks]
        chroma_metadatas = [clean_metadata_for_chroma(chunk["metadata"]) for chunk in chroma_chunks]

        collection_name = f"paper_{sanitized_name}_paragraphs"
        print(f"🏷️  Nombre colección Chroma: {collection_name}")

        chroma_store = Chroma.from_texts(
            texts=chroma_texts,
            metadatas=chroma_metadatas,
            embedding=embeddings,
            collection_name=collection_name
        )

        # FAISS
        faiss_texts = [chunk["text"] for chunk in faiss_chunks]
        faiss_metadatas = [chunk["metadata"] for chunk in faiss_chunks]

        faiss_store = FAISS.from_texts(
            texts=faiss_texts,
            metadatas=faiss_metadatas,
            embedding=embeddings
        )

        print("✅ Vectorstores creados")

    except Exception as e:
        print(f"❌ Error creando vectorstores: {e}")
        return {"status": "error", "paper_name": paper_name, "error": f"Error vectorstores: {e}"}

    # 7. Guardar todo
    print("💾 Guardando archivos...")
    paper_dir = os.path.join(output_dir, sanitized_name)  # Usar nombre sanitizado para carpeta también
    os.makedirs(paper_dir, exist_ok=True)

    try:
        # Guardar FAISS
        faiss_path = os.path.join(paper_dir, "faiss_index")
        faiss_store.save_local(faiss_path)

        # Guardar Chroma persistente
        chroma_path = os.path.join(paper_dir, "chroma_db")
        chroma_persistent = Chroma(
            collection_name=collection_name,
            embedding_function=embeddings,
            persist_directory=chroma_path
        )
        chroma_persistent.add_texts(texts=chroma_texts, metadatas=chroma_metadatas)

        # Guardar datos procesados
        data_to_save = {
            "paper_name": paper_name,
            "paper_title": paper_title,  # ✅ LÍNEA AÑADIDA
            "sanitized_name": sanitized_name,
            "collection_name": collection_name,
            "sections_data": sections_data,
            "sections": sections,
            "resolved_references": resolved_references,
            "metadata": metadata,
            "chroma_chunks_count": len(chroma_chunks),
            "faiss_chunks_count": len(faiss_chunks)
        }

        data_path = os.path.join(paper_dir, "processed_data.pkl")
        with open(data_path, 'wb') as f:
            pickle.dump(data_to_save, f)

        print(f"💾 Guardado en: {paper_dir}")

        return {
            "status": "success",
            "paper_name": paper_name,
            "paper_title": paper_title,  # ✅ LÍNEA AÑADIDA
            "sanitized_name": sanitized_name,
            "collection_name": collection_name,
            "sections_count": len(sections),
            "references_count": len(resolved_references),
            "chroma_chunks": len(chroma_chunks),
            "faiss_chunks": len(faiss_chunks),
            "output_dir": paper_dir
        }

    except Exception as e:
        print(f"❌ Error guardando: {e}")
        return {"status": "error", "paper_name": paper_name, "error": f"Error guardando: {e}"}

# ============================================================================
# PROCESAMIENTO MASIVO
# ============================================================================

def process_all_papers(papers_dir: str, output_base_dir: str) -> Dict[str, Any]:
    """Procesa todos los papers en una carpeta"""

    print("\n🚀 INICIANDO PROCESAMIENTO MASIVO DE PAPERS")
    print("=" * 60)

    # Verificar que las APIs estén configuradas
    if 'embeddings' not in globals() or 'gemini_model' not in globals():
        print("❌ APIs no configuradas. Ejecuta primero la celda de configuración.")
        return {"status": "error", "message": "APIs no configuradas"}

    print("✅ Usando APIs ya configuradas")

    # Buscar PDFs
    pdf_files = list(Path(papers_dir).glob("*.pdf"))
    print(f"📚 Papers encontrados: {len(pdf_files)}")

    if not pdf_files:
        return {"status": "error", "message": "No se encontraron archivos PDF"}

    # Crear directorio de salida
    os.makedirs(output_base_dir, exist_ok=True)

    # Procesar cada paper
    results = []
    successful = 0
    failed = 0

    for i, pdf_path in enumerate(pdf_files, 1):
        print(f"\n📄 [{i}/{len(pdf_files)}] {pdf_path.name}")
        print("-" * 40)

        result = process_single_paper(str(pdf_path), output_base_dir)
        results.append(result)

        if result["status"] == "success":
            successful += 1
            print(f"✅ Éxito: {result['paper_name']}")
            print(f"📋 Título: {result.get('paper_title', 'No disponible')}")  # ✅ LÍNEA AÑADIDA

        else:
            failed += 1
            print(f"❌ Fallo: {result['paper_name']} - {result['error']}")

    # Guardar resumen general
    summary = {
        "total_papers": len(pdf_files),
        "successful": successful,
        "failed": failed,
        "success_rate": f"{(successful/len(pdf_files)*100):.1f}%",
        "results": results,
        "output_directory": output_base_dir,
        "papers_with_titles": [  # ✅ SECCIÓN AÑADIDA
            {
                "paper_name": r["paper_name"],
                "title": r.get("paper_title", "Título no disponible"),
                "status": r["status"]
            }
            for r in results if r["status"] == "success"
        ]
    }

    summary_path = os.path.join(output_base_dir, "processing_summary.json")
    with open(summary_path, 'w', encoding='utf-8') as f:  # ✅ AÑADIR encoding='utf-8'
        json.dump(summary, f, indent=2, ensure_ascii=False)  # ✅ AÑADIR ensure_ascii=False

    print("\n" + "=" * 60)
    print("📊 RESUMEN FINAL")
    print("=" * 60)
    print(f"📚 Total papers: {summary['total_papers']}")
    print(f"✅ Exitosos: {summary['successful']}")
    print(f"❌ Fallidos: {summary['failed']}")
    print(f"📈 Tasa éxito: {summary['success_rate']}")
    print(f"📁 Directorio salida: {output_base_dir}")
    print(f"📋 Resumen guardado en: {summary_path}")

    # Mostrar títulos procesados
    if summary["papers_with_titles"]:
        print(f"\n📋 TÍTULOS EXTRAÍDOS:")
        for paper in summary["papers_with_titles"]:
            print(f"  • {paper['paper_name']}: {paper['title']}")

    return summary

In [6]:
# ============================================================================
# EJECUTAR PROCESAMIENTO
# ============================================================================

# Configuración de rutas
papers_dir = "/content/drive/MyDrive/MsC Tesis/Final: Scientific references/papersEVALS"
output_base_dir = "/content/drive/MyDrive/MsC Tesis/Final: Scientific references/vectores/multi_papers"

print("🎯 CONFIGURACIÓN:")
print(f"📁 Papers dir: {papers_dir}")
print(f"📁 Output dir: {output_base_dir}")

# Ejecutar procesamiento
summary = process_all_papers(papers_dir, output_base_dir)

if summary and summary.get("status") != "error":
    print("\n🎉 ¡PROCESAMIENTO COMPLETADO!")
    print(f"📊 Revisa los resultados en: {summary['output_directory']}")
else:
    print("\n❌ Procesamiento falló.")
    if summary:
        print(f"Error: {summary.get('message', 'Error desconocido')}")

🎯 CONFIGURACIÓN:
📁 Papers dir: /content/drive/MyDrive/MsC Tesis/Final: Scientific references/papersEVALS
📁 Output dir: /content/drive/MyDrive/MsC Tesis/Final: Scientific references/vectores/multi_papers

🚀 INICIANDO PROCESAMIENTO MASIVO DE PAPERS
✅ Usando APIs ya configuradas
📚 Papers encontrados: 29

📄 [1/29] paper_078_CAMEL_ Communicative Agents for Mind Exploration.pdf
----------------------------------------

🔄 Procesando: paper_078_CAMEL_ Communicative Agents for Mind Exploration
📝 Nombre sanitizado: paper_078_CAMEL_Communicative_Agents_for_Mind_Expl
📊 Páginas: 54, Caracteres: 130568
📋 Extrayendo título del paper...
📋 Título extraído: Algorithm of Thoughts:
Enhancing Exploration of Ideas in Large Language Models
🧠 Iniciando análisis con Gemini...
📝 Prompts creados, enviando a Gemini...
  📋 Solicitando secciones...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 2254.51ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 4917.08ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1850.10ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1091.43ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1091.75ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 998.11ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 790.30ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:ge

  📋 Respuesta de secciones recibida, procesando...
  📋 Secciones procesadas: 17
  📚 Solicitando referencias...
  📚 Respuesta de referencias recibida, procesando...
  📚 Referencias procesadas: 59
✅ Análisis completado - Secciones: 17, Referencias: 59
📄 Procesando secciones...
✂️ Creando chunks...
📦 Chunks Chroma: 67, FAISS: 17
🔄 Creando vectorstores...
🏷️  Nombre colección Chroma: paper_paper_078_CAMEL_Communicative_Agents_for_Mind_Expl_paragraphs
✅ Vectorstores creados
💾 Guardando archivos...
💾 Guardado en: /content/drive/MyDrive/MsC Tesis/Final: Scientific references/vectores/multi_papers/paper_078_CAMEL_Communicative_Agents_for_Mind_Expl
✅ Éxito: paper_078_CAMEL_ Communicative Agents for Mind Exploration
📋 Título: Algorithm of Thoughts:
Enhancing Exploration of Ideas in Large Language Models

📄 [2/29] paper_079_Tree of Thoughts_ Deliberate Problem Solving with .pdf
----------------------------------------

🔄 Procesando: paper_079_Tree of Thoughts_ Deliberate Problem Solving with 
📝 N

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 2541.58ms


  📋 Respuesta de secciones recibida, procesando...
  📋 Secciones procesadas: 22
  📚 Solicitando referencias...
  📚 Respuesta de referencias recibida, procesando...
  📚 Referencias procesadas: 122
✅ Análisis completado - Secciones: 22, Referencias: 122
📄 Procesando secciones...
✂️ Creando chunks...
📦 Chunks Chroma: 50, FAISS: 3
🔄 Creando vectorstores...
🏷️  Nombre colección Chroma: paper_paper_086_Learning_Transferable_Visual_Models_From_paragraphs
✅ Vectorstores creados
💾 Guardando archivos...
💾 Guardado en: /content/drive/MyDrive/MsC Tesis/Final: Scientific references/vectores/multi_papers/paper_086_Learning_Transferable_Visual_Models_From
✅ Éxito: paper_086_Learning Transferable Visual Models From Natural L
📋 Título: Learning Transferable Visual Models From Natural Language Supervision

📄 [10/29] paper_087_Flamingo_ a Visual Language Model for Few-Shot Lea.pdf
----------------------------------------

🔄 Procesando: paper_087_Flamingo_ a Visual Language Model for Few-Shot Lea
📝 Nombre

...

  📚 Respuesta de referencias recibida, procesando...
  📚 Referencias procesadas: 76
✅ Análisis completado - Secciones: 46, Referencias: 76
📄 Procesando secciones...
✂️ Creando chunks...
📦 Chunks Chroma: 146, FAISS: 100
🔄 Creando vectorstores...
🏷️  Nombre colección Chroma: paper_paper_096_PaLM_Scaling_Language_Modeling_with_Path_paragraphs
✅ Vectorstores creados
💾 Guardando archivos...
💾 Guardado en: /content/drive/MyDrive/MsC Tesis/Final: Scientific references/vectores/multi_papers/paper_096_PaLM_Scaling_Language_Modeling_with_Path
✅ Éxito: paper_096_PaLM_ Scaling Language Modeling with Pathways
📋 Título: PaLM: Scaling Language Modeling with Pathways

📄 [19/29] paper_094_FreshLLMs_ Refreshing Large Language Models with S.pdf
----------------------------------------

🔄 Procesando: paper_094_FreshLLMs_ Refreshing Large Language Models with S
📝 Nombre sanitizado: paper_094_FreshLLMs_Refreshing_Large_Language_Mode
📊 Páginas: 21, Caracteres: 71723
📋 Extrayendo título del paper...
📋 Título 

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1023.39ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 39957.57ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 2408.21ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 5793.71ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1321.99ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1168.66ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1220.22ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash

  📋 Respuesta de secciones recibida, procesando...
  📋 Secciones procesadas: 18
  📚 Solicitando referencias...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 4504.18ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1826.79ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1979.07ms


  📚 Respuesta de referencias recibida, procesando...
  📚 Referencias procesadas: 130
✅ Análisis completado - Secciones: 18, Referencias: 130
📄 Procesando secciones...
✂️ Creando chunks...
📦 Chunks Chroma: 91, FAISS: 29
🔄 Creando vectorstores...
🏷️  Nombre colección Chroma: paper_paper_100_T5_Text-to-Text_Transfer_Transformer_paragraphs
✅ Vectorstores creados
💾 Guardando archivos...
💾 Guardado en: /content/drive/MyDrive/MsC Tesis/Final: Scientific references/vectores/multi_papers/paper_100_T5_Text-to-Text_Transfer_Transformer
✅ Éxito: paper_100_T5_ Text-to-Text Transfer Transformer
📋 Título: Language Models are Few-Shot Learners

📄 [25/29] paper_096_LLaMA 2_ Open Foundation and Fine-Tuned Chat Model.pdf
----------------------------------------

🔄 Procesando: paper_096_LLaMA 2_ Open Foundation and Fine-Tuned Chat Model
📝 Nombre sanitizado: paper_096_LLaMA_2_Open_Foundation_and_Fine-Tuned_C
📊 Páginas: 77, Caracteres: 263888
📋 Extrayendo título del paper...
📋 Título extraído: Llama 2: Open